# Data Lake V1 + optional Silver V2 — Google Colab

Run **Runtime → Run all** in order from `main`. Connect the Google account that owns or has access to the shared Data Lake. The notebook is repeatable: it **verifies and skips completed immutable snapshots**, but never silently overwrites them.

Pipeline: Clone/install → mount shared Drive → Bronze → Silver V1 → Silver V2 (dry-run by default) → Gold V1 from explicitly selected Silver snapshot.

**Required outside Colab:** The Drive owner must share the Data Lake folder with each teammate. Teammates should add its shortcut to their **My Drive** as `vietnamese-multilabel-topic-classification`. Alternatively set `DATA_LAKE_ROOT` to the mounted folder path before execution. Cloning GitHub alone does *not* grant Drive access. Each person must explicitly authorize `drive.mount()`.

For previewing a feature branch, set `VTC_GIT_REF` in the first cell. Default is `main`. For V1 including modeling, prefer `notebooks/v1/colab_end_to_end_v1.ipynb`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_colab_project")
if (PROJECT_DIR / ".git").is_dir():
    status = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True
    )
    if status.strip():
        raise RuntimeError("Colab checkout is dirty; do not discard local modifications")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project path exists but does not contain a Git repository")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"],
    cwd=PROJECT_DIR,
    check=True,
)
SRC_DIR = (PROJECT_DIR / "src").resolve()
sys.path[:] = [p for p in sys.path if p != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))

# A Colab kernel can preserve imports across git checkout and editable reinstall.
import importlib

prefix = "vietnamese_topic_classification"
for name in sorted(
    [key for key in sys.modules if key == prefix or key.startswith(prefix + ".")],
    key=len, reverse=True,
):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.config import load_domain_config
from vietnamese_topic_classification.data.bronze import BronzeReader
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldPipeline, GoldReader
from vietnamese_topic_classification.data.silver import SilverReader
from vietnamese_topic_classification.data.silver.v1 import SilverPipeline
from vietnamese_topic_classification.data.silver.v2 import SilverV2Pipeline

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True,
).strip()
print("Repo:", BRANCH, "@", revision)


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Shared Data Lake:", DATA_LAKE_ROOT)


In [ ]:
# Pinned input provenance. Keep these equal when reusing the same snapshot.
DATA_CONFIG = load_domain_config()
SAMPLE_SIZE = 50_000
SEED = 42
SHUFFLE_BUFFER = 10_000

# Safe default: V2 is analyzed, but NOT written unless explicitly approved.
PUBLISH_SILVER_V2 = False
GOLD_SOURCE_VERSION = "v1"  # Change to "v2" only when a V2 snapshot is published.
BUILD_GOLD = True

if GOLD_SOURCE_VERSION not in ("v1", "v2"):
    raise ValueError("GOLD_SOURCE_VERSION must be v1 or v2")

SOURCE_REVISION = DATA_CONFIG["dataset"]["revision"]
STAMP = f"{SOURCE_REVISION[:10]}_seed{SEED}_buf{SHUFFLE_BUFFER}_n{SAMPLE_SIZE}"
MANIFEST_NAME = f"source_manifest_{STAMP}.json"
BRONZE_PATH = DATA_LAKE_ROOT / "01_bronze" / f"vtsnlp_raw_sample_{STAMP}.jsonl"
BRONZE_MANIFEST = DATA_LAKE_ROOT / "00_source_metadata" / MANIFEST_NAME
BRONZE_REPORT = DATA_LAKE_ROOT / "04_reports" / f"bronze_ingestion_{STAMP}.json"
print("Source:", DATA_CONFIG["dataset"]["id"], SOURCE_REVISION)


In [ ]:
# One-time ingestion from pinned VTSNLP dataset. Existing data are verified,
# not re-downloaded or overwritten.
bronze_existing = [p for p in (BRONZE_PATH, BRONZE_MANIFEST, BRONZE_REPORT) if p.exists()]
if bronze_existing:
    if len(bronze_existing) != 3:
        raise FileExistsError(f"Partial Bronze publication: {bronze_existing}")
    bronze_reader = BronzeReader(DATA_LAKE_ROOT, MANIFEST_NAME)
    bronze_verified = bronze_reader.verify()
    if bronze_verified["records"] != SAMPLE_SIZE:
        raise AssertionError("Bronze sample size does not match configuration")
    print("BRONZE VERIFIED — SKIPPED INGEST:", bronze_verified)
else:
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/01_ingest_bronze.py"),
        "--output-root", str(DATA_LAKE_ROOT),
        "--sample-size", str(SAMPLE_SIZE),
        "--seed", str(SEED),
        "--shuffle-buffer", str(SHUFFLE_BUFFER),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    bronze_reader = BronzeReader(DATA_LAKE_ROOT, MANIFEST_NAME)
    bronze_verified = bronze_reader.verify()
    print("BRONZE INGESTED AND VERIFIED:", bronze_verified)


In [ ]:
# V1 layout: 02_silver/v1 and 04_reports/silver/v1.
silver_v1_pipeline = SilverPipeline(bronze_reader)
silver_v1_paths = silver_v1_pipeline.planned_paths()
v1_existing = [p for p in silver_v1_paths.values() if p.exists()]
v1_legacy = [
    DATA_LAKE_ROOT / "02_silver" / silver_v1_paths["silver"].name,
    DATA_LAKE_ROOT / "02_silver" / silver_v1_paths["rejected"].name,
    DATA_LAKE_ROOT / "04_reports" / silver_v1_paths["report"].name,
]
if any(p.exists() for p in v1_legacy):
    raise FileExistsError("Legacy V1 snapshot found outside v1/; inspect/migrate before publishing")
if v1_existing and len(v1_existing) != len(silver_v1_paths):
    raise FileExistsError(f"Partial Silver V1 publication: {v1_existing}")
if not v1_existing:
    silver_v1_pipeline.run()

v1_reader = SilverReader(DATA_LAKE_ROOT, silver_v1_paths["report"].name)
v1_verified = v1_reader.verify()
if v1_reader.report.get("source_sha256") != bronze_verified["sha256"]:
    raise AssertionError("Silver V1 lineage differs from verified Bronze")
print("SILVER V1 VERIFIED:", v1_verified)


In [ ]:
# V2 does not re-ingest Bronze. A two-pass corpus analysis is performed
# from the verified V1 snapshot; no new Drive file during dry-run.
silver_v2_pipeline = SilverV2Pipeline(v1_reader)
v2_paths = silver_v2_pipeline.planned_paths(v1_verified["sha256"])
v2_existing = [p for p in v2_paths.values() if p.exists()]
if v2_existing and len(v2_existing) != len(v2_paths):
    raise FileExistsError(f"Partial Silver V2 publication: {v2_existing}")

if v2_existing:
    v2_reader = SilverReader(DATA_LAKE_ROOT, v2_paths["report"].name)
    v2_verified = v2_reader.verify()
    print("EXISTING SILVER V2 VERIFIED:", v2_verified)
else:
    pilot = silver_v2_pipeline.run(dry_run=True)
    print("SILVER V2 DRY RUN:", {
        key: pilot["report"][key]
        for key in ("cleaned_rows", "review_rows", "guardrail_rows", "removed_total_chars")
    })
    if PUBLISH_SILVER_V2:
        silver_v2_pipeline.run()
        v2_reader = SilverReader(DATA_LAKE_ROOT, v2_paths["report"].name)
        v2_verified = v2_reader.verify()
        print("SILVER V2 PUBLISHED AND VERIFIED:", v2_verified)


In [ ]:
# Gold V1 reads exactly the selected Silver snapshot.
if BUILD_GOLD:
    if GOLD_SOURCE_VERSION == "v2":
        if not v2_paths["report"].is_file():
            raise FileNotFoundError(
                "No published Silver V2 snapshot. Set PUBLISH_SILVER_V2=True and "
                "verify the V2 data first, or choose GOLD_SOURCE_VERSION='v1'."
            )
        source_reader = SilverReader(DATA_LAKE_ROOT, v2_paths["report"].name)
    else:
        source_reader = v1_reader
    source_verified = source_reader.verify()
    gold_pipeline = GoldPipeline(source_reader, config=GoldConfig(seed=SEED))
    gold_paths = gold_pipeline.planned_paths(source_verified["sha256"])
    existing_gold = [p for p in gold_paths.values() if p.exists()]
    if existing_gold:
        if len(existing_gold) != len(gold_paths):
            raise FileExistsError(f"Partial Gold V1 publication: {existing_gold}")
        gold_reader = GoldReader(DATA_LAKE_ROOT, gold_paths["manifest"].name)
        gold_verified = gold_reader.verify()
        if gold_reader.manifest["source"]["silver_sha256"] != source_verified["sha256"]:
            raise AssertionError("Gold source SHA256 differs from selected Silver snapshot")
        print("GOLD V1 VERIFIED — SKIPPED BUILD:", gold_verified)
    else:
        gold_pipeline.run()
        gold_reader = GoldReader(DATA_LAKE_ROOT, gold_paths["manifest"].name)
        gold_verified = gold_reader.verify()
        print("GOLD V1 PUBLISHED AND VERIFIED:", gold_verified)
else:
    print("Gold step disabled by BUILD_GOLD=False")


## After running

Review the printed checksums and snapshot paths. Silver V2 only publishes when `PUBLISH_SILVER_V2=True`; V1 and Gold automatically skip already-verified snapshots. Modeling is intentionally separate in `notebooks/v1/colab_modeling_runner.ipynb` so training never starts implicitly. If a collaborator only has read permission on a shared Drive, all existing-snapshot verification will work, but new publications require write permission. Do not attempt simultaneous publishing of the same snapshot from multiple users.